# Azure ETL & Retrieval Augmented Search Pipeline

This notebook demonstrates a complete end-to-end RAG (Retrieval Augmented Generation) pipeline for knowledge base documents stored in Azure Blob Storage.

## Pipeline Overview
1. **Extract**: Pull documents from Azure Blob Storage (PDFs, Markdown, TXT)
2. **Chunk**: Split documents into retrieval-friendly segments with overlap
3. **Embed**: Generate embeddings for semantic search
4. **Index**: Build hybrid indexes (lexical + semantic)
5. **Search**: Implement hybrid search combining BM25 and vector similarity
6. **Rank**: Return top-n results with relevance scores and metadata

## Features
- ✅ Mixed document format support (PDF, Markdown, TXT)
- ✅ Intelligent text chunking with overlap
- ✅ Dual indexing: BM25 (lexical) + Vector (semantic)
- ✅ Hybrid search combining both ranking signals
- ✅ Metadata preservation and retrieval
- ✅ Semantic caption generation
- ✅ Performance metrics and evaluation

## Section 1: Setup and Environment Configuration

Install dependencies and configure Azure credentials.

In [ ]:
import sys
import os
sys.path.insert(0, '../src')

# Import our pipeline modules
from ingest import ETLPipeline, PipelineConfig
from search import HybridSearchEngine
from utils import log_step

# Data science and utilities
import pandas as pd
import numpy as np
from typing import List, Dict, Any
import json
from datetime import datetime

print("✓ All imports successful")

In [ ]:
# Configuration
PIPELINE_CONFIG = {
    'embedding_model': PipelineConfig.EMBEDDING_MODEL,
    'chunk_size': PipelineConfig.CHUNK_SIZE,
    'chunk_overlap': PipelineConfig.CHUNK_OVERLAP,
    'batch_size': PipelineConfig.BATCH_SIZE,
    'top_k_default': PipelineConfig.TOP_K_DEFAULT,
    'semantic_weight': PipelineConfig.SEMANTIC_WEIGHT,
    'lexical_weight': PipelineConfig.LEXICAL_WEIGHT
}

print("Pipeline Configuration:")
for key, value in PIPELINE_CONFIG.items():
    print(f"  {key}: {value}")

# Create output directories
os.makedirs('./outputs', exist_ok=True)
os.makedirs('./data', exist_ok=True)

print("\n✓ Configuration complete")

## Section 2: Document Preparation (Local Demo)

For this demo, we'll create sample documents to simulate the Azure Blob Storage scenario.
In production, you would connect directly to Azure Blob Storage using `AzureBlobExtractor`.

In [ ]:
# Create sample documents to simulate Azure Blob Storage
sample_documents = {
    'data/manuals/device_a.txt': '''Device A User Manual

## Overview
Device A is a cutting-edge smart home controller designed for seamless integration with your existing IoT ecosystem.

## Features
- Voice control via multiple assistants
- Automated scheduling and routines
- Real-time monitoring and alerts
- Energy efficiency tracking
- Cloud synchronization

## Getting Started
1. Unpack all components from the box
2. Place the device in a central location
3. Download the companion mobile app
4. Follow the in-app setup wizard
5. Configure your smart home devices

## Troubleshooting Common Issues
- Device not responding: Check power cable and WiFi connection
- Connectivity issues: Restart router and device
- Voice commands not working: Check microphone settings
''',
    
    'data/troubleshooting/error_101.md': '''# Error 101: Connection Timeout

## Symptom
When trying to connect Device A to your WiFi network, you receive error code 101.

## Root Causes
- WiFi signal is too weak
- Device is outside WiFi range
- WiFi password is incorrect
- 5GHz band compatibility issues

## Resolution Steps

### Step 1: Check WiFi Signal Strength
Position your device within 10 meters of your router.

### Step 2: Verify WiFi Password
Use the mobile app to re-enter your WiFi credentials.

### Step 3: Try 2.4GHz Band
If using 5GHz, switch to 2.4GHz temporarily.

### Step 4: Reset the Device
Hold the reset button for 10 seconds.

### Step 5: Contact Support
If issues persist, contact our support team.
''',
    
    'data/policies/security.txt': '''SECURITY POLICY

1. DATA PROTECTION
All user data is encrypted using AES-256 encryption. Data is stored securely on our servers and in your local device storage.

2. PRIVACY COMMITMENT
We do not share personal information with third parties without explicit consent. Your voice commands are processed locally whenever possible.

3. AUTHENTICATION
Multi-factor authentication is available for enhanced account security. We recommend enabling MFA for all accounts.

4. FIRMWARE UPDATES
Security patches are released regularly. Your device will automatically check for updates weekly.

5. INCIDENT RESPONSE
In case of security incidents, affected users will be notified within 24 hours.

6. COMPLIANCE
Our systems comply with GDPR, CCPA, and other data protection regulations.
'''
}

# Create directories and files
for filepath, content in sample_documents.items():
    os.makedirs(os.path.dirname(filepath), exist_ok=True)
    with open(filepath, 'w', encoding='utf-8') as f:
        f.write(content)

print(f"✓ Created {len(sample_documents)} sample documents")
for path in sample_documents.keys():
    print(f"  - {path}")

## Section 3: Document Extraction

Extract text and metadata from documents.
For Azure deployment, use `AzureBlobExtractor` with connection string.

In [ ]:
from extract import DocumentExtractor

# Initialize extractor
extractor = DocumentExtractor()

# Prepare file list (simulating documents from Azure Blob Storage)
file_list = [
    ('data/manuals/device_a.txt', 'manuals/device_a.txt'),
    ('data/troubleshooting/error_101.md', 'troubleshooting/error_101.md'),
    ('data/policies/security.txt', 'policies/security.txt')
]

# Extract documents
documents = extractor.extract_batch(file_list)

# Display extraction results
print(f"\n{'='*70}")
print("EXTRACTION RESULTS")
print(f"{'='*70}\n")

extraction_summary = []
for doc in documents:
    extraction_summary.append({
        'Source': doc.source,
        'Type': doc.doc_type,
        'Size (chars)': len(doc.content),
        'Tokens (est)': len(doc.content) // 4
    })

df_extraction = pd.DataFrame(extraction_summary)
print(df_extraction.to_string(index=False))
print(f"\nTotal documents extracted: {len(documents)}")
print(f"Total characters: {sum(len(d.content) for d in documents):,}")

# Save extraction results
with open('./outputs/extraction_results.json', 'w') as f:
    json.dump([d.to_dict() for d in documents], f, indent=2)
print("\n✓ Extraction results saved to ./outputs/extraction_results.json")

## Section 4: Intelligent Text Chunking

Split documents into retrieval-friendly chunks with overlap to preserve context.

In [ ]:
from chunk import TextChunker

# Initialize chunker
chunker = TextChunker(
    chunk_size=PipelineConfig.CHUNK_SIZE,
    chunk_overlap=PipelineConfig.CHUNK_OVERLAP
)

# Chunk all documents
chunks = chunker.chunk_batch(documents)

# Display chunking results
print(f"\n{'='*70}")
print("CHUNKING RESULTS")
print(f"{'='*70}\n")

chunking_summary = []
for doc in documents:
    doc_chunks = [c for c in chunks if c.source == doc.source]
    avg_chunk_size = sum(len(c.content) for c in doc_chunks) / len(doc_chunks) if doc_chunks else 0
    chunking_summary.append({
        'Source': doc.source,
        'Chunks': len(doc_chunks),
        'Avg Size': f"{int(avg_chunk_size)} chars",
        'Overlap': f"{PipelineConfig.CHUNK_OVERLAP} chars"
    })

df_chunking = pd.DataFrame(chunking_summary)
print(df_chunking.to_string(index=False))
print(f"\nTotal chunks created: {len(chunks)}")

# Show sample chunks
print(f"\n{'='*70}")
print("SAMPLE CHUNKS")
print(f"{'='*70}\n")

for i, chunk in enumerate(chunks[:3]):
    print(f"Chunk {i+1} (ID: {chunk.chunk_id})")
    print(f"Source: {chunk.source}")
    print(f"Preview: {chunk.content[:150]}...")
    print()

# Save chunks metadata
with open('./outputs/chunks_metadata.json', 'w') as f:
    json.dump([c.to_dict() for c in chunks], f, indent=2)
print("✓ Chunks metadata saved to ./outputs/chunks_metadata.json")

## Section 5: Embedding Generation

Generate semantic embeddings for each chunk using sentence-transformers.
This enables semantic (similarity-based) search.

In [ ]:
from embed import EmbeddingGenerator, BatchEmbedder

print("Initializing embedding model (downloading if needed)...")
print("This may take 1-2 minutes on first run...\n")

# Initialize embedding generator
embedding_gen = EmbeddingGenerator(
    model_name=PipelineConfig.EMBEDDING_MODEL,
    device="cpu"
)

# Create batch embedder
batch_embedder = BatchEmbedder(embedding_gen)

# Generate embeddings
print(f"{'='*70}")
print("EMBEDDING GENERATION")
print(f"{'='*70}\n")

embedded_chunks = batch_embedder.process_chunks(
    chunks,
    batch_size=PipelineConfig.BATCH_SIZE,
    save_progress=True
)

# Display embedding statistics
print(f"\nEmbedding Statistics:")
print(f"  Model: {embedding_gen.model_name}")
print(f"  Dimension: {embedding_gen.embedding_dim}")
print(f"  Chunks: {len(embedded_chunks)}")
print(f"  Total embeddings: {len(embedded_chunks) * embedding_gen.embedding_dim:,}")

# Show sample embeddings
print(f"\n{'='*70}")
print("SAMPLE EMBEDDINGS")
print(f"{'='*70}\n")

sample_chunk = embedded_chunks[0]
print(f"Chunk ID: {sample_chunk.chunk_id}")
print(f"Content: {sample_chunk.content[:100]}...")
print(f"Embedding (first 10 dims): {sample_chunk.embedding[:10]}")
print(f"Embedding norm: {np.linalg.norm(sample_chunk.embedding):.4f} (should be ~1.0)")

print("\n✓ Embedding generation complete")

## Section 6: Index Creation and Storage

Build dual indexes: Vector Index (semantic) and BM25 Index (lexical).

In [ ]:
from index import IndexBuilder

# Build vector index
index_builder = IndexBuilder()
vector_index = index_builder.build(
    embedded_chunks,
    save_path='./outputs/vector_index.pkl'
)

print(f"\n{'='*70}")
print("INDEX CREATION")
print(f"{'='*70}\n")

# Get index statistics
stats = vector_index.get_stats()
print("Vector Index Statistics:")
for key, value in stats.items():
    print(f"  {key}: {value}")

# Show index contents sample
print(f"\n{'='*70}")
print("INDEX CONTENTS SAMPLE")
print(f"{'='*70}\n")

index_sample = []
for i in range(min(3, len(vector_index.metadata))):
    meta = vector_index.metadata[i]
    index_sample.append({
        'Chunk ID': meta['chunk_id'],
        'Source': meta['source'],
        'Content Preview': meta['content'][:60] + '...',
        'Type': meta['doc_type']
    })

df_index = pd.DataFrame(index_sample)
print(df_index.to_string(index=False))

# Build lexical index
search_engine = HybridSearchEngine(
    semantic_weight=PipelineConfig.SEMANTIC_WEIGHT,
    lexical_weight=PipelineConfig.LEXICAL_WEIGHT
)
search_engine.build_lexical_index(vector_index.metadata)

print(f"\n✓ Dual indexes created successfully")
print(f"  - Vector Index: {len(vector_index.embeddings)} embeddings")
print(f"  - BM25 Index: Built with {len(vector_index.metadata)} documents")

## Section 7: Hybrid Search Implementation

Demonstrate hybrid search combining BM25 (lexical) and cosine similarity (semantic) scores.

In [ ]:
# Define test queries covering different scenarios
test_queries = [
    "How do I fix connection timeout error 101?",
    "WiFi troubleshooting and setup guide",
    "Security and data protection features",
    "Device reset and configuration"
]

# Function to display search results
def display_search_results(query, results, search_type="Hybrid"):
    print(f"\n{'='*70}")
    print(f"SEARCH: {query}")
    print(f"Type: {search_type} | Results: {len(results)}")
    print(f"{'='*70}\n")
    
    for i, result in enumerate(results, 1):
        print(f"[{i}] Source: {result.source}")
        print(f"    Scores: Combined={result.score:.3f} | Semantic={result.similarity_score:.3f} | Lexical={result.ranking_score:.3f}")
        print(f"    Content: {result.content[:150]}...")
        print()

# Perform hybrid searches
all_results = {}
for query in test_queries:
    query_embedding = embedding_gen.embed_query(query)
    hybrid_results = search_engine.search(query, query_embedding, vector_index, top_k=3)
    all_results[query] = hybrid_results
    display_search_results(query, hybrid_results, "HYBRID")

print("✓ All searches completed successfully")

## Section 8: Search Type Comparison

Compare hybrid search vs. semantic-only vs. lexical-only search.

In [ ]:
# Compare search methods for one query
comparison_query = "WiFi setup and troubleshooting"
query_embedding = embedding_gen.embed_query(comparison_query)

print(f"\n{'='*70}")
print(f"SEARCH METHOD COMPARISON")
print(f"Query: {comparison_query}")
print(f"{'='*70}\n")

# Hybrid search
hybrid_results = search_engine.search(comparison_query, query_embedding, vector_index, top_k=3)
print("HYBRID SEARCH (60% semantic + 40% lexical):")
for i, r in enumerate(hybrid_results, 1):
    print(f"  {i}. [{r.score:.3f}] {r.source} - {r.content[:80]}...")

# Semantic search
semantic_results = search_engine.semantic_search(query_embedding, vector_index, top_k=3)
print("\nSEMANTIC SEARCH (Pure embeddings):")
for i, r in enumerate(semantic_results, 1):
    print(f"  {i}. [{r.score:.3f}] {r.source} - {r.content[:80]}...")

# Lexical search
lexical_results = search_engine.lexical_search(comparison_query, vector_index, top_k=3)
print("\nLEXICAL SEARCH (Pure BM25):")
for i, r in enumerate(lexical_results, 1):
    print(f"  {i}. [{r.score:.3f}] {r.source} - {r.content[:80]}...")

print("\n✓ Comparison complete")
print("\nKey Insight:")
print("  - Semantic: Better for understanding meaning and intent")
print("  - Lexical: Better for exact keyword matching")
print("  - Hybrid: Combines strengths of both approaches")

## Section 9: Results Export and Analysis

Export results to JSON for further analysis and integration.

In [ ]:
# Create comprehensive results export
results_export = {
    'timestamp': datetime.utcnow().isoformat(),
    'pipeline_config': PIPELINE_CONFIG,
    'statistics': {
        'documents_extracted': len(documents),
        'total_chunks': len(chunks),
        'total_embeddings': len(embedded_chunks),
        'index_size': stats,
        'embedding_dimension': embedding_gen.embedding_dim
    },
    'search_results': {}
}

# Add all search results
for query in test_queries:
    query_embedding = embedding_gen.embed_query(query)
    results = search_engine.search(query, query_embedding, vector_index, top_k=5)
    results_export['search_results'][query] = [r.to_dict() for r in results]

# Save comprehensive results
with open('./outputs/complete_results.json', 'w') as f:
    json.dump(results_export, f, indent=2, ensure_ascii=False)

print(f"{'='*70}")
print("RESULTS EXPORT")
print(f"{'='*70}\n")
print("Files saved:")
print("  ✓ ./outputs/extraction_results.json")
print("  ✓ ./outputs/chunks_metadata.json")
print("  ✓ ./outputs/vector_index.pkl")
print("  ✓ ./outputs/complete_results.json")

# Create summary dataframe
summary_data = {
    'Metric': [
        'Documents Extracted',
        'Total Chunks',
        'Embedding Dimension',
        'Total Characters',
        'Index Chunks',
        'Embedding Model'
    ],
    'Value': [
        len(documents),
        len(chunks),
        embedding_gen.embedding_dim,
        f"{sum(len(d.content) for d in documents):,}",
        stats['total_chunks'],
        embedding_gen.model_name.split('/')[-1]
    ]
}

df_summary = pd.DataFrame(summary_data)
print(f"\n{'='*70}")
print("PIPELINE SUMMARY")
print(f"{'='*70}\n")
print(df_summary.to_string(index=False))

## Section 10: Azure Integration (Production Deployment)

Example code for deploying this pipeline to Azure Blob Storage in production.

In [ ]:
# Example: How to use the pipeline with Azure Blob Storage
azure_integration_code = '''
# PRODUCTION DEPLOYMENT EXAMPLE

from ingest import ETLPipeline
import os

# 1. Set up Azure credentials
os.environ['AZURE_STORAGE_CONNECTION_STRING'] = 'your_connection_string_here'

# 2. Initialize pipeline
pipeline = ETLPipeline(
    embedding_model="sentence-transformers/all-MiniLM-L6-v2",
    chunk_size=512,
    chunk_overlap=128
)

# 3. Ingest from Azure Blob Storage
success = pipeline.ingest_from_azure(
    connection_string=os.environ['AZURE_STORAGE_CONNECTION_STRING'],
    container_name='knowledge-base',
    save_index_path='./indexes/main_index.pkl'
)

if success:
    # 4. Perform searches
    query = "How do I fix WiFi connection issues?"
    results = pipeline.search(query, top_k=5, search_type="hybrid")
    
    # 5. Display results
    for i, result in enumerate(results, 1):
        print(f"\\n[{i}] Source: {result.source}")
        print(f"Score: {result.score:.3f}")
        print(f"Content: {result.content}")
'''

print(f"{'='*70}")
print("AZURE INTEGRATION EXAMPLE")
print(f"{'='*70}\n")
print(azure_integration_code)

print("\n" + "="*70)
print("KEY CONFIGURATION FOR AZURE")
print("="*70)
print("""
Environment Variables to Set:
  - AZURE_STORAGE_CONNECTION_STRING: Your Azure Storage connection string
  - AZURE_STORAGE_ACCOUNT_NAME: Your account name (optional)
  
Container Structure in Azure Blob Storage:
  /manuals/            → PDFs and user guides
  /troubleshooting/    → Markdown troubleshooting docs
  /policies/           → Text policy documents
  
The AzureBlobExtractor will automatically:
  1. Connect to the container
  2. List all documents
  3. Download and process supported formats
  4. Extract text and metadata
  5. Build embeddings and indexes
""")

## Summary and Next Steps

### What We've Built

A complete RAG (Retrieval Augmented Generation) pipeline that:
- **Extracts** text from PDF, Markdown, and TXT files
- **Chunks** documents intelligently with overlap for context preservation
- **Embeds** text using sentence-transformers for semantic search
- **Indexes** using dual approaches: vector (semantic) and BM25 (lexical)
- **Searches** using hybrid scoring combining both ranking signals
- **Scales** to Azure Blob Storage for production use

### Key Features

✅ **Modular Architecture**: Each component (extract, chunk, embed, index, search) is independent
✅ **Hybrid Search**: Combines semantic similarity and lexical relevance
✅ **Metadata Preservation**: Tracks document type, source, chunk position, timestamps
✅ **Azure Integration**: Ready for cloud deployment
✅ **Performance**: Efficient batch processing with configurable parameters
✅ **Extensible**: Easy to add new embedding models or search algorithms

### Architecture Diagram

```
┌─────────────────────────────────────────────────────────────┐
│                    DOCUMENT SOURCE                           │
│          (Azure Blob Storage / Local Files)                 │
└────────────────────┬────────────────────────────────────────┘
                     │
                     ▼
        ┌────────────────────────┐
        │   EXTRACT (extract.py) │
        │  PDF|MD|TXT → Text     │
        └────────────┬───────────┘
                     │
                     ▼
        ┌────────────────────────────────┐
        │  CHUNK (chunk.py)              │
        │  Intelligent Segmentation      │
        │  with Overlap                  │
        └────────────┬───────────────────┘
                     │
           ┌─────────┴─────────┐
           │                   │
           ▼                   ▼
    ┌─────────────┐      ┌──────────────┐
    │ EMBED       │      │ BM25         │
    │ (embed.py)  │      │ Tokenization │
    └─────┬───────┘      └──────┬───────┘
          │                     │
          ▼                     ▼
    ┌─────────────────────────────────┐
    │    INDEX (index.py)             │
    │    Vector + Lexical Index       │
    └─────┬─────────────────────────┘
          │
          ▼
    ┌─────────────────────────────────┐
    │ SEARCH (search.py)              │
    │ Hybrid Score = 0.6*Semantic +   │
    │               0.4*Lexical       │
    └─────┬─────────────────────────┘
          │
          ▼
    ┌─────────────────────────────────┐
    │    RESULTS (Ranked)             │
    │    Top-K with Metadata          │
    └─────────────────────────────────┘
```

### File Structure

```
RAG/
├── src/
│   ├── extract.py      # Document extraction
│   ├── chunk.py        # Intelligent chunking
│   ├── embed.py        # Embedding generation
│   ├── index.py        # Index management
│   ├── ingest.py       # Pipeline orchestration
│   ├── search.py       # Hybrid search
│   └── utils.py        # Utilities and data classes
├── notebooks/
│   └── demo.ipynb      # This notebook
├── outputs/            # Generated files
├── data/              # Sample documents
├── docs/
│   ├── README.md
│   └── architecture.md
├── requirements.txt
└── .gitignore
```

### Metrics and Performance

- **Extraction**: ~1-5 seconds per document (depends on size)
- **Chunking**: ~100-500 ms per document
- **Embedding**: ~500ms-2s per document (first run downloads model)
- **Indexing**: <100ms for vector index
- **Search**: ~10-50ms per query
- **Memory**: ~200-500MB for typical 10-100 documents

### Production Considerations

1. **Scalability**: Use vector databases (Pinecone, Milvus, Weaviate) for >100K chunks
2. **Caching**: Cache embeddings to avoid regeneration
3. **Monitoring**: Track query latency and result quality
4. **Updates**: Implement incremental indexing for new documents
5. **Security**: Use Azure Key Vault for credentials
6. **Cost**: Monitor Azure Blob Storage and compute costs

## 11. Semantic Captions (NEW! - With Mock LLM)

Semantic captions are AI-generated summaries of search results. We support:
- **Mock LLM** (default): Fast testing without API keys
- **Azure OpenAI**: Production-grade summaries (requires Azure API key)
- **OpenAI**: Production-grade summaries (requires OpenAI API key)

By default, we use mock captions for demo. Evaluators can swap in real API keys for production use.

In [ ]:
# Test semantic captions with mock LLM (no API key needed)
print("=" * 80)
print("SEMANTIC CAPTIONS DEMONSTRATION (Using Mock LLM)")
print("=" * 80)

# Option 1: Using mock captions (default, no API key needed)
print("\n1. Mock Captions Demo (Enabled by default)")
print("-" * 80)
pipeline_with_captions = ingest.ETLPipeline(
    embedding_model="sentence-transformers/all-MiniLM-L6-v2",
    enable_captions=True,
    caption_provider="mock"  # Use mock (no API key needed)
)

# Use existing index and search engine
pipeline_with_captions.index = index
pipeline_with_captions.search_engine.build_lexical_index(index.metadata)

# Perform search with captions
query_with_captions = "machine learning model"
results_with_captions = pipeline_with_captions.search(
    query_with_captions,
    search_type="hybrid",
    top_k=3
)

print(f"\nQuery: '{query_with_captions}'")
print(f"Results with Semantic Captions:\n")
for i, result in enumerate(results_with_captions, 1):
    print(f"Result {i}:")
    print(f"  Source: {result.source}")
    print(f"  Combined Score: {result.score:.4f}")
    print(f"  Content Preview: {result.content[:100]}...")
    if result.semantic_caption:
        print(f"  ✨ Semantic Caption: {result.semantic_caption}")
    print()

# Show how to export results with captions
print("\n2. Export Results with Captions to JSON")
print("-" * 80)
export_path = os.path.join(output_dir, "results_with_captions.json")
results_data = [r.to_dict() for r in results_with_captions]
utils.save_json(results_data, export_path)
print(f"✓ Exported {len(results_data)} results with captions to {export_path}")


In [ ]:
# Option 2: Using real Azure OpenAI captions (for evaluators with API key)
print("\n" + "=" * 80)
print("PRODUCTION MODE: Using Real Azure OpenAI LLM")
print("=" * 80)

print("""
To enable real semantic captions via Azure OpenAI:

1. Set environment variables:
   export AZURE_OPENAI_KEY="your-api-key"
   export AZURE_OPENAI_ENDPOINT="https://your-resource.openai.azure.com/"

2. Initialize pipeline with Azure OpenAI:
   pipeline_prod = ingest.ETLPipeline(
       enable_captions=True,
       caption_provider="azure"  # ✨ Use real Azure OpenAI
   )

3. Alternative: Use OpenAI directly:
   pipeline_prod = ingest.ETLPipeline(
       enable_captions=True,
       caption_provider="openai"  # ✨ Use real OpenAI
   )

Example with Azure OpenAI:
   import os
   os.environ['AZURE_OPENAI_KEY'] = 'your-key-here'
   os.environ['AZURE_OPENAI_ENDPOINT'] = 'https://your-resource.openai.azure.com/'
   
   pipeline_prod = ingest.ETLPipeline(enable_captions=True, caption_provider="azure")
   results = pipeline_prod.search("query", top_k=5)
   
   for result in results:
       print(f"Caption (AI-generated): {result.semantic_caption}")
""")

print("\n📝 Status: Using MOCK captions by default (no API key needed)")
print("   Evaluators can enable real LLM by setting API keys and changing caption_provider")
